In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

## RAG 절차
- 1. docx문서를 읽는다(python-docx) 
- 2. 읽어온 문서를 쪼갠다(text_spliter이용)
    * 모델의 context window와 비용 확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임베딩 -> vector Store에 저장 => chroma(local 파일)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성(LangChain)

# 0. 패키지 설치

In [2]:
# 문서읽어오기
#%pip install -q docx2txt

In [11]:
# 텍스트를 chunk로 나누는 기능 
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 파일 저장 chroma (cf. chromadb와 다름)
%pip install -q langchain-chroma

In [9]:
# langchain 사용
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기(x)

In [14]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

CPU times: total: 4 s
Wall time: 3.99 s


1

# 2. 문서를 쪼개면서 읽기(o)
## 2.1 1500토큰씩 겹쳐서 쪼개면서 읽기

In [16]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
# gpt-4o-mini, .... 모델의 토큰 추출 방식 : cl100k_base방식
# gpt-4,  gpt4 turbo .. 모델의 토큰 추출방식 : o200k_base방식
text_splitter = TokenTextSplitter(
        encoding_name="cl100k_base", # 토큰을 세는 방식 이름
        chunk_size=1500,             # 1500토큰 수 기준으로 chunk 분리
        chunk_overlap=200,
        # separator 파라미터가 없어
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 겹쳐 쪼개면서 읽어오는 시간 :', runtime)

문서를 겹쳐 쪼개면서 읽어오는 시간 : 6.033790826797485


In [24]:
# 각 chunk별 글자수 : len(documents[0].page_content)
print([len(document.page_content) for document in documents] )

[1718, 1657, 1639, 1649, 1738, 1455, 1280, 1534, 1326, 1613, 1596, 1590, 1575, 1633, 1622, 1561, 1615, 1643, 1574, 1475, 1423, 1607, 1450, 1500, 1641, 1602, 1538, 1649, 1672, 1595, 1603, 1671, 1588, 1636, 1599, 1542, 1564, 1600, 1697, 1716, 1660, 1632, 1632, 1656, 1664, 1604, 1484, 1670, 1704, 1647, 1584, 1528, 1625, 1625, 1636, 1621, 1575, 1625, 1600, 1544, 1578, 1612, 1604, 1672, 1692, 1639, 1669, 1624, 1626, 1627, 1615, 1611, 1608, 1681, 1702, 1689, 1605, 1541, 1570, 1733, 2048, 1592, 1629, 1582, 1616, 1605, 1622, 1612, 1596, 1674, 1693, 1720, 1657, 1706, 1527, 1552, 1584, 1610, 1675, 1693, 1669, 1623, 1569, 1715, 1689, 1629, 1554, 1623, 1691, 1677, 1683, 1615, 1657, 1591, 1482, 1531, 1589, 1690, 1568, 1527, 1449, 1577, 1555, 1444, 1540, 1536, 1611, 1684, 1711, 1667, 1646, 1621, 1742, 1583, 1634, 1623, 1613, 1662, 1560, 1632, 1543, 1645, 1529, 1535, 1603, 1595, 1615, 1641, 1556, 1512, 1685, 1629, 1612, 1593, 1612, 1576, 1666, 1604, 1629, 1575, 1602, 1614, 1516, 1569, 1587, 1608, 162

In [27]:
# chunk 평균 글자수, 최소 글자수, 최대 글자수
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))

1611.5555555555557
2048
1280


## 2.2 1500글짜씩 겹쳐 쪼개 읽어오기

In [ ]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter